In [ ]:
pip install transformers

In [ ]:
pip install "transformers[torch]"

In [5]:
import pandas as pd
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

In [39]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [15]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [16]:
train_data.shape

(14732, 3)

In [10]:
val_data.shape

(818, 3)

In [11]:
train_data.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [12]:
train_data.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
14727    False
14728    False
14729    False
14730    False
14731    False
Length: 14732, dtype: bool

### random sampling

In [40]:
train_data = train_data.sample(n=4000,random_state=42).reset_index(drop=True)

In [41]:
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [42]:
train_data.shape

(4000, 3)

In [20]:
val_data.shape

(500, 3)

## Data Preprocessing

In [21]:
## we use regex
import re

In [22]:
def clean_data(text):
    text = re.sub(r"\r\n"," ",text) ## Remove next lines
    text = re.sub(r"<.*?>"," ",text) ## remove html tags
    text =re.sub(r"\s+"," ",text) ## remove white space
    text = text.strip().lower()

    return text
    

In [43]:
train_data.columns

Index(['id', 'dialogue', 'summary'], dtype='object')

In [44]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [45]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet: claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [46]:
train_data["summary"][0]

"violet sent claire austin's article."

### Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [47]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    target= tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = target["input_ids"] ## in input labels adds targets 
    return inputs

In [48]:
train_data = train_data.apply(tokenize,axis=1).tolist()
val_data = val_data.apply(tokenize,axis=1).tolist()

In [49]:
train_data[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [50]:
val_data[0]

{'input_ids': [3, 15, 26, 26, 10, 2275, 210, 6, 410, 25, 1616, 24, 79, 31, 60, 3, 23560, 178, 12, 3, 9, 315, 3066, 58, 4659, 10, 14228, 9, 9, 9, 9, 144, 3, 10, 32, 4659, 10, 150, 55, 213, 31, 26, 25, 1616, 24, 58, 3, 15, 26, 26, 10, 168, 6, 34, 31, 7, 882, 2314, 3, 15, 26, 26, 10, 11, 13515, 131, 1219, 178, 4659, 10, 11, 103, 25, 214, 125, 34, 1112, 21, 178, 58, 3, 15, 26, 26, 10, 79, 751, 31, 17, 483, 8, 5812, 7, 3, 15, 26, 26, 10, 68, 3, 23, 214, 8, 16879, 56, 129, 7873, 972, 4659, 10, 11, 3, 23, 3382, 24, 19, 3, 9, 888, 24, 19, 5741, 12, 143, 762, 1842, 3, 15, 26, 26, 10, 17945, 6, 3382, 78, 3, 15, 26, 26, 10, 79, 43, 3, 9, 6613, 194, 13, 1705, 3, 31, 235, 143, 378, 1842, 31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 